# Optimisation des modèles de churn

La validation croisée et SMOTE sont encapsulés dans le pipeline. Le test final reste séparé et n'est utilisé qu'après la sélection des hyperparamètres.

In [1]:
from pathlib import Path
import joblib
import pandas as pd

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
)
from xgboost import XGBClassifier

ROOT = Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"
X_train = pd.read_csv(PROCESSED / "X_train_cluster.csv")
X_test = pd.read_csv(PROCESSED / "X_test_cluster.csv")
y_train = pd.read_csv(PROCESSED / "y_train.csv").squeeze().astype(int)
y_test = pd.read_csv(PROCESSED / "y_test.csv").squeeze().astype(int)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
negative, positive = y_train.value_counts().sort_index().tolist()
scale_pos_weight = negative / positive
print(f"Train: {X_train.shape}, test: {X_test.shape}")
print(y_train.value_counts(normalize=True).round(3))

Train: (5634, 51), test: (1409, 51)
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


In [2]:
baseline_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, solver="liblinear", class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=1,
    ),
}

def make_pipeline(model):
    return Pipeline([
        ("smote", SMOTE(random_state=42)),
        ("model", model),
    ])

cv_rows = []
for name, model in baseline_models.items():
    scores = cross_validate(
        make_pipeline(model),
        X_train,
        y_train,
        cv=cv,
        scoring={"f1": "f1", "roc_auc": "roc_auc", "recall": "recall"},
        n_jobs=-1,
    )
    cv_rows.append({
        "model": name,
        "stage": "baseline",
        "f1_mean": scores["test_f1"].mean(),
        "f1_std": scores["test_f1"].std(),
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "recall_mean": scores["test_recall"].mean(),
    })

baseline_cv = pd.DataFrame(cv_rows).sort_values("f1_mean", ascending=False)
baseline_cv.round(4)

,model,stage,f1_mean,f1_std,roc_auc_mean,recall_mean
0,Logistic Regression,baseline,0.6386,0.0266,0.8440,0.7258
2,XGBoost,baseline,0.6282,0.0192,0.8457,0.7933
1,Random Forest,baseline,0.5763,0.0211,0.8277,0.5251


In [3]:
search_spaces = {
    "Logistic Regression": (
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            solver="liblinear",
            random_state=42,
        ),
        {
            "model__C": [0.01, 0.1, 1, 10, 100],
            "model__solver": ["liblinear"],
        },
    ),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
        {
            "model__n_estimators": [150, 250, 350],
            "model__max_depth": [None, 8, 15, 25],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", "log2", None],
        },
    ),
    "XGBoost": (
        XGBClassifier(
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=1,
        ),
        {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [3, 5, 7],
            "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
            "model__subsample": [0.8, 1.0],
        },
    ),
}

searches = {}
optimization_rows = []
for name, (model, params) in search_spaces.items():
    search = RandomizedSearchCV(
        make_pipeline(model),
        params,
        n_iter=10,
        scoring="f1",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    optimization_rows.append({
        "model": name,
        "stage": "optimized",
        "f1_mean": search.best_score_,
        "best_params": search.best_params_,
    })

optimized_cv = pd.DataFrame(optimization_rows).sort_values("f1_mean", ascending=False)
optimized_cv[["model", "stage", "f1_mean"]].round(4)

/home/alaric/Desktop/studies/projects/client-churn-prediction/.venv/lib/python3.12/site-packages/sklearn/model_selection/_search.py:327: UserWarning: The total space of parameters 5 is smaller than n_iter=10. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/home/alaric/Desktop/studies/projects/client-churn-prediction/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/alaric/Desktop/studies/projects/client-churn-prediction/.venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/

,model,stage,f1_mean
0,Logistic Regression,optimized,0.6386
2,XGBoost,optimized,0.6303
1,Random Forest,optimized,0.6180


In [4]:
def evaluate(model, X, y):
    prediction = model.predict(X)
    probability = model.predict_proba(X)[:, 1]
    return {
        "precision": precision_score(y, prediction, zero_division=0),
        "recall": recall_score(y, prediction, zero_division=0),
        "f1": f1_score(y, prediction, zero_division=0),
        "roc_auc": roc_auc_score(y, probability),
        "confusion_matrix": confusion_matrix(y, prediction).tolist(),
    }

test_rows = []
for name, model in baseline_models.items():
    fitted = make_pipeline(model).fit(X_train, y_train)
    test_rows.append({"model": name, "stage": "baseline", **evaluate(fitted, X_test, y_test)})
for name, search in searches.items():
    test_rows.append({
        "model": name,
        "stage": "optimized",
        **evaluate(search.best_estimator_, X_test, y_test),
    })

test_comparison = pd.DataFrame(test_rows)
test_comparison.drop(columns="confusion_matrix").sort_values("f1", ascending=False).round(4)

,model,stage,precision,recall,f1,roc_auc
2,XGBoost,baseline,0.5190,0.8021,0.6303,0.8413
5,XGBoost,optimized,0.5092,0.8182,0.6277,0.8391
4,Random Forest,optimized,0.5608,0.6658,0.6088,0.8244
1,Random Forest,baseline,0.5722,0.5936,0.5827,0.8230
0,Logistic Regression,baseline,0.2752,1.0000,0.4316,0.8407
3,Logistic Regression,optimized,0.2752,1.0000,0.4316,0.8407


In [5]:
best_name = test_comparison.sort_values("f1", ascending=False).iloc[0]["model"]
best_model = searches[best_name].best_estimator_
best_metrics = test_comparison[
    (test_comparison["model"] == best_name)
    & (test_comparison["stage"] == "optimized")
].iloc[0].to_dict()
print("Best model:", best_name)
print(classification_report(y_test, best_model.predict(X_test), zero_division=0))
print("Best parameters:", searches[best_name].best_params_)

output = PROCESSED
baseline_cv.to_csv(output / "optimization_baseline_cv.csv", index=False)
optimized_cv.to_csv(output / "optimization_search_results.csv", index=False)
test_comparison.to_csv(output / "optimization_test_comparison.csv", index=False)
joblib.dump(best_model, output / "optimized_model.joblib")
joblib.dump({
    "model": best_model,
    "model_name": best_name,
    "metrics": best_metrics,
    "best_params": searches[best_name].best_params_,
}, output / "optimized_model_bundle.joblib")
print("Saved optimized model artifacts to", output)

Best model: XGBoost
              precision    recall  f1-score   support

           0       0.92      0.71      0.80      1035
           1       0.51      0.82      0.63       374

    accuracy                           0.74      1409
   macro avg       0.71      0.77      0.72      1409
weighted avg       0.81      0.74      0.76      1409

Best parameters: {'model__subsample': 0.8, 'model__n_estimators': 300, 'model__max_depth': 5, 'model__learning_rate': 0.01}
Saved optimized model artifacts to /home/alaric/Desktop/studies/projects/client-churn-prediction/data/processed
